## Part-of-Speech (POS) Tagging

**Part-of-Speech (POS) tagging** is the process of marking up a word in a text (corpus) as corresponding to a particular part of speech, based on both its definition and its context—that is, its relationship with adjacent and related words in a phrase, sentence, or paragraph.

For example:

*   `"The dog barks loudly." `
    *   `The` (determiner)
    *   `dog` (noun)
    *   `barks` (verb)
    *   `loudly` (adverb)

POS tagging is a fundamental step in many Natural Language Processing (NLP) tasks, such as sentiment analysis, machine translation, and information extraction.

## Hidden Markov Models (HMMs) for POS Tagging

A **Hidden Markov Model (HMM)** is a statistical Markov model in which the system being modeled is assumed to be a Markov process with unobserved (hidden) states.

In the context of POS tagging:

1.  **States (Hidden)**: These are the Part-of-Speech tags (e.g., Noun, Verb, Adjective).
2.  **Observations (Visible)**: These are the actual words in the sentence.

An HMM for POS tagging works by defining two main types of probabilities:

*   **Transition Probabilities**: The probability of moving from one tag to another (e.g., `P(Verb | Noun)`).
    *   `P(Tag_i | Tag_{i-1})`
*   **Emission Probabilities**: The probability of a word being generated given a specific tag (e.g., `P(dog | Noun)`).
    *   `P(Word_i | Tag_i)`

Given a sequence of words (observations), the goal is to find the most likely sequence of tags (hidden states). This is typically done using the **Viterbi Algorithm**.

### Setup and Data Loading

First, let's set up our environment by importing the necessary NLTK libraries and downloading the Brown corpus, which is a common dataset for POS tagging.

In [1]:
import nltk
from nltk.corpus import brown
from collections import defaultdict

# Download necessary NLTK data
# 'punkt' for tokenization, 'brown' for the corpus, 'universal_tagset' for simplified tags
nltk.download('punkt')
nltk.download('brown')
nltk.download('universal_tagset')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


True

In [2]:
# Load the Brown corpus with the universal tagset
# The universal tagset simplifies the tags (e.g., Noun, Verb, Adj, Adv, Det, etc.)

# Get tagged sentences. Each sentence is a list of (word, tag) tuples.
tagged_sentences = brown.tagged_sents(tagset='universal')

print(f"Number of tagged sentences in Brown corpus: {len(tagged_sentences)}")
print("\nFirst tagged sentence example:")
print(tagged_sentences[0])

# Get a list of all unique words and tags
all_words = [word.lower() for sent in tagged_sentences for word, tag in sent]
all_tags = [tag for sent in tagged_sentences for word, tag in sent]

unique_words = sorted(list(set(all_words)))
unique_tags = sorted(list(set(all_tags)))

print(f"\nNumber of unique words: {len(unique_words)}")
print(f"Number of unique tags: {len(unique_tags)}")
print(f"Unique tags: {unique_tags}")

Number of tagged sentences in Brown corpus: 57340

First tagged sentence example:
[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]

Number of unique words: 49815
Number of unique tags: 12
Unique tags: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']


### Splitting Data into Training and Testing Sets

We'll split the corpus into a training set (e.g., 90%) to estimate the HMM probabilities and a testing set (e.g., 10%) to evaluate the model's performance.

In [3]:
# Define the split ratio
split_ratio = 0.9
split_point = int(len(tagged_sentences) * split_ratio)

train_sentences = tagged_sentences[:split_point]
test_sentences = tagged_sentences[split_point:]

print(f"Number of sentences for training: {len(train_sentences)}")
print(f"Number of sentences for testing: {len(test_sentences)}")

Number of sentences for training: 51606
Number of sentences for testing: 5734


### Calculating Probabilities (Transition and Emission)

To build our HMM, we need to calculate two main types of probabilities from our training data:

1.  **Transition Probabilities (P(Tag_j | Tag_i))**: The probability of a tag `Tag_j` following a tag `Tag_i`.
2.  **Emission Probabilities (P(Word_k | Tag_i))**: The probability of a word `Word_k` being emitted given a tag `Tag_i`.

We will use `defaultdict` to store these counts and then convert them into probabilities.

In [4]:
# Initialize dictionaries for counts
tag_counts = defaultdict(lambda: 0)
tag_tag_counts = defaultdict(lambda: 0) # C(tag_i, tag_j)
word_tag_counts = defaultdict(lambda: 0) # C(word, tag)

# Populate counts from training data
for sent in train_sentences:
    # Add a 'START' tag to mark the beginning of a sentence
    # This helps in calculating the probability of the first tag in a sentence
    previous_tag = 'START'
    for word, tag in sent:
        word = word.lower() # Convert words to lowercase for consistency

        tag_counts[tag] += 1
        word_tag_counts[(word, tag)] += 1
        tag_tag_counts[(previous_tag, tag)] += 1

        previous_tag = tag

    # Add an 'END' tag to mark the end of a sentence (optional, but good practice)
    tag_tag_counts[(previous_tag, 'END')] += 1 # Transition from last tag to END

# --- Calculate Probabilities ---

# Transition Probabilities P(tag_j | tag_i) = C(tag_i, tag_j) / C(tag_i)
transition_probabilities = defaultdict(lambda: defaultdict(lambda: 0.0))

# Add 'START' tag to tag_counts for normalization of first tag transitions
tag_counts['START'] = len(train_sentences)

for (prev_tag, current_tag), count in tag_tag_counts.items():
    if tag_counts[prev_tag] > 0:
        transition_probabilities[prev_tag][current_tag] = count / tag_counts[prev_tag]


# Emission Probabilities P(word | tag) = C(word, tag) / C(tag)
emission_probabilities = defaultdict(lambda: defaultdict(lambda: 0.0))

for (word, tag), count in word_tag_counts.items():
    if tag_counts[tag] > 0:
        emission_probabilities[tag][word] = count / tag_counts[tag]

print("Probability calculation complete.")

# Display a few examples
print("\n--- Example Probabilities ---")
print(f"P('NOUN' | 'DET'): {transition_probabilities['DET']['NOUN']:.4f}")
print(f"P('VERB' | 'NOUN'): {transition_probabilities['NOUN']['VERB']:.4f}")
print(f"P('dog' | 'NOUN'): {emission_probabilities['NOUN']['dog']:.4f}")
print(f"P('run' | 'VERB'): {emission_probabilities['VERB']['run']:.4f}")


Probability calculation complete.

--- Example Probabilities ---
P('NOUN' | 'DET'): 0.6242
P('VERB' | 'NOUN'): 0.1580
P('dog' | 'NOUN'): 0.0003
P('run' | 'VERB'): 0.0008


### Viterbi Algorithm for POS Tagging

The **Viterbi algorithm** is a dynamic programming algorithm for finding the most likely sequence of hidden states—the Viterbi path—that results in a sequence of observed events, especially in the context of Hidden Markov Models.

In our case:
*   **Observed events**: The words in a sentence.
*   **Hidden states**: The POS tags corresponding to those words.

The algorithm works by building a trellis (or table) where each cell `V[t][s]` stores the probability of the most likely path ending at state `s` at time `t`, given the observation sequence up to time `t`.

The steps generally involve:
1.  **Initialization**: Calculate the probability of the first tag for the first word.
2.  **Recursion**: For each subsequent word, calculate the probability of each possible tag, considering the transition probabilities from the previous tags and the emission probability of the current word.
3.  **Termination**: Find the path with the highest probability for the last word.
4.  **Path Backtracking**: Reconstruct the sequence of tags from the highest probability path.

In [5]:
def viterbi(words, unique_tags, transition_probabilities, emission_probabilities):
    # Initialize Viterbi path probability and backpointer tables
    V = defaultdict(lambda: defaultdict(lambda: 0.0)) # V[t][tag] = max probability of a tag sequence ending with 'tag' at time t
    backpointer = defaultdict(lambda: defaultdict(lambda: '')) # backpointer[t][tag] = previous tag in the most likely path to 'tag' at time t

    # All possible tags, including 'START' for initial state
    all_tags = list(unique_tags) + ['START', 'END']

    # 1. Initialization step (t = 0)
    # The first word's tag depends on the probability of starting with a tag and emitting the first word
    first_word = words[0].lower()
    for tag in unique_tags:
        # P(tag | START) * P(first_word | tag)
        trans_prob = transition_probabilities['START'].get(tag, 1e-10) # Use a small value for unseen transitions
        emiss_prob = emission_probabilities[tag].get(first_word, 1e-10) # Use a small value for unseen words
        V[0][tag] = trans_prob * emiss_prob
        backpointer[0][tag] = 'START'

    # 2. Recursion step (t > 0)
    for t in range(1, len(words)):
        current_word = words[t].lower()
        for current_tag in unique_tags:
            max_prob = 0.0
            best_prev_tag = ''
            for prev_tag in unique_tags:
                # V[t-1][prev_tag] * P(current_tag | prev_tag) * P(current_word | current_tag)
                prob = V[t-1][prev_tag] * \
                       transition_probabilities[prev_tag].get(current_tag, 1e-10) * \
                       emission_probabilities[current_tag].get(current_word, 1e-10)

                if prob > max_prob:
                    max_prob = prob
                    best_prev_tag = prev_tag
            V[t][current_tag] = max_prob
            backpointer[t][current_tag] = best_prev_tag

    # 3. Termination step (find the best path ending with 'END')
    max_prob = 0.0
    best_last_tag = ''
    last_word_idx = len(words) - 1
    for tag in unique_tags:
        # V[last_word_idx][tag] * P(END | tag)
        prob = V[last_word_idx][tag] * transition_probabilities[tag].get('END', 1e-10)
        if prob > max_prob:
            max_prob = prob
            best_last_tag = tag

    # 4. Path Backtracking (reconstruct the best tag sequence)
    best_tag_sequence = [''] * len(words)
    current_tag = best_last_tag
    for t in range(len(words) - 1, -1, -1):
        best_tag_sequence[t] = current_tag
        current_tag = backpointer[t][current_tag]

    return list(zip(words, best_tag_sequence)), max_prob

print("Viterbi algorithm defined.")

# --- Test the Viterbi algorithm with a sample sentence ---
sample_sentence = "The dog barks loudly .".split()
predicted_tags, prob = viterbi(sample_sentence, unique_tags, transition_probabilities, emission_probabilities)

print("\n--- Sample Sentence POS Tagging ---")
print(f"Sentence: {' '.join(sample_sentence)}")
print(f"Predicted Tags: {predicted_tags}")
print(f"Probability of path: {prob}")

sample_sentence_2 = "I am learning machine learning.".split()
predicted_tags_2, prob_2 = viterbi(sample_sentence_2, unique_tags, transition_probabilities, emission_probabilities)

print("\n--- Another Sample Sentence POS Tagging ---")
print(f"Sentence: {' '.join(sample_sentence_2)}")
print(f"Predicted Tags: {predicted_tags_2}")
print(f"Probability of path: {prob_2}")

Viterbi algorithm defined.

--- Sample Sentence POS Tagging ---
Sentence: The dog barks loudly .
Predicted Tags: [('The', 'DET'), ('dog', 'NOUN'), ('barks', 'VERB'), ('loudly', 'ADV'), ('.', '.')]
Probability of path: 1.9747476422233728e-22

--- Another Sample Sentence POS Tagging ---
Sentence: I am learning machine learning.
Predicted Tags: [('I', 'PRON'), ('am', 'VERB'), ('learning', 'VERB'), ('machine', 'NOUN'), ('learning.', '.')]
Probability of path: 2.5082214327340744e-25


### Evaluating the HMM POS Tagger

Now, let's evaluate the performance of our HMM-based POS tagger on the held-out `test_sentences`. We will calculate the accuracy by comparing the predicted tags from our `viterbi` function with the actual tags in the test set.

In [6]:
correct_tags = 0
total_tags = 0

# Iterate through each sentence in the test set
# For simplicity, we'll process each test sentence as if it were a new, untagged sentence
for i, sent in enumerate(test_sentences):
    # Extract words and true tags for the current sentence
    words = [word for word, tag in sent]
    true_tags = [tag for word, tag in sent]

    # Use the Viterbi algorithm to predict tags
    # We need to handle potential errors or empty predictions gracefully
    try:
        predicted_tagged_sentence, _ = viterbi(words, unique_tags, transition_probabilities, emission_probabilities)
        predicted_tags = [tag for word, tag in predicted_tagged_sentence]
    except Exception as e:
        # In case of any error during Viterbi for a sentence, skip it or log the error
        print(f"Error processing sentence {i}: {e}")
        predicted_tags = [] # Assume no tags predicted

    # Compare predicted tags with true tags
    if len(predicted_tags) == len(true_tags):
        for pred_tag, true_tag in zip(predicted_tags, true_tags):
            if pred_tag == true_tag:
                correct_tags += 1
            total_tags += 1
    else:
        # This might happen if the Viterbi algorithm fails for some words
        # or if there's an issue with how unique_tags is handled.
        # For evaluation, we only count where lengths match to avoid index errors.
        print(f"Warning: Length mismatch for sentence {i}. Predicted {len(predicted_tags)}, True {len(true_tags)}")

# Calculate overall accuracy
accuracy = correct_tags / total_tags if total_tags > 0 else 0

print(f"\n--- HMM POS Tagger Evaluation ---")
print(f"Total tags evaluated: {total_tags}")
print(f"Correctly predicted tags: {correct_tags}")
print(f"Accuracy on test set: {accuracy:.4f}")


--- HMM POS Tagger Evaluation ---
Total tags evaluated: 95442
Correctly predicted tags: 90486
Accuracy on test set: 0.9481
